# Отчет по лабораторной работе №3
## Тема: Арифметические вычисления

**Студент:** ИУ5-35Б Штыка Денис Александрович  
**Вариант:** 1  
**Выражение:** $(((c-b)-b)/b)$

## Задание 1
**Условие:** Написать программу на ассемблере, которая получает символ как параметр командной строки и выводит его ASCII-код.

**Исходный код (`task1.asm`):**
```asm
format ELF64
public _start

section '.data' writable
    newline db 10

section '.bss' writable
    result_buf rb 32

section '.text' executable
_start:
    cmp qword [rsp], 2
    jne .error

    mov rsi, [rsp + 16]
    cmp byte [rsi + 1], 0
    jne .error
    movzx rax, byte [rsi]
    cmp rax, 127
    ja .error

    call print_number
    xor rdi, rdi
    jmp exit

.error:
    mov rdi, 1

exit:
    mov rax, 60
    syscall

print_number:
    mov r8, result_buf + 32
    mov r9, r8
    mov rcx, 10

.convert:
    xor rdx, rdx
    div rcx
    dec r8
    add dl, '0'
    mov [r8], dl
    test rax, rax
    jnz .convert

    mov rax, 1
    mov rdi, 1
    mov rsi, r8
    mov rdx, r9
    sub rdx, r8
    syscall

    mov rax, 1
    mov rdi, 1
    mov rsi, newline
    mov rdx, 1
    syscall
    ret
```

In [ ]:
symbol = 'A'
ascii_code = ord(symbol)
assert ascii_code == 65
print(f'{symbol} -> {ascii_code}')

## Задание 2
**Условие:** Разработать программу на ассемблере, вычисляющую арифметическое выражение по аргументам командной строки `a`, `b` и `c`.

Для варианта 1 задано $(((c-b)-b)/b)$. Переменная `a` в выражении не используется. Вычисления целочисленные, деление округляется к нулю; `b` должно быть ненулевым.

**Исходный код (`task2.asm`):**
```asm
format ELF64
public _start

section '.data' writable
    newline db 10

section '.bss' writable
    a_value rq 1
    b_value rq 1
    c_value rq 1
    result_buf rb 32

section '.text' executable
_start:
    cmp qword [rsp], 4
    jne .error

    mov rsi, [rsp + 16]
    call parse_integer
    mov [a_value], rax

    mov rsi, [rsp + 24]
    call parse_integer
    mov [b_value], rax
    test rax, rax
    jz .error

    mov rsi, [rsp + 32]
    call parse_integer
    mov [c_value], rax

    mov rax, [c_value]
    sub rax, [b_value]
    sub rax, [b_value]
    cqo
    idiv qword [b_value]
    call print_number

    xor rdi, rdi
    jmp exit

.error:
    mov rdi, 1

exit:
    mov rax, 60
    syscall

parse_integer:
    xor rax, rax
    mov r8, 1
    cmp byte [rsi], '-'
    jne .digits
    mov r8, -1
    inc rsi

.digits:
    movzx rdx, byte [rsi]
    test dl, dl
    jz .parsed
    sub rdx, '0'
    imul rax, rax, 10
    add rax, rdx
    inc rsi
    jmp .digits

.parsed:
    imul rax, r8
    ret

print_number:
    xor r10, r10
    test rax, rax
    jns .absolute
    mov r10, 1
    neg rax

.absolute:
    mov r8, result_buf + 32
    mov r9, r8
    mov rcx, 10

.convert:
    xor rdx, rdx
    div rcx
    dec r8
    add dl, '0'
    mov [r8], dl
    test rax, rax
    jnz .convert

    test r10, r10
    jz .write
    dec r8
    mov byte [r8], '-'

.write:
    mov rax, 1
    mov rdi, 1
    mov rsi, r8
    mov rdx, r9
    sub rdx, r8
    syscall

    mov rax, 1
    mov rdi, 1
    mov rsi, newline
    mov rdx, 1
    syscall
    ret
```

In [ ]:
def truncating_division(numerator, denominator):
    quotient = abs(numerator) // abs(denominator)
    return -quotient if (numerator < 0) != (denominator < 0) else quotient

samples = [(1, 2, 7, 1), (1, 2, 1, -1)]
for a, b, c, expected in samples:
    result = truncating_division((c - b) - b, b)
    assert result == expected
    print(f'a={a}, b={b}, c={c}: результат = {result}')

## Задание 3
**Условие:** Написать на языке C программу, аналогичную заданию 2, и сравнить результаты.

**Исходный код (`task3.c`):**
```c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[]) {
    if (argc != 4) {
        return 1;
    }

    long long a = strtoll(argv[1], NULL, 10);
    long long b = strtoll(argv[2], NULL, 10);
    long long c = strtoll(argv[3], NULL, 10);
    (void)a;

    if (b == 0) {
        return 1;
    }

    long long result = ((c - b) - b) / b;
    printf("%lld\n", result);
    return 0;
}
```

## Сборка и запуск

FASM формирует ELF object-файл, который затем связывается командой `ld`. Из корня репозитория выполните:

```sh
fasm Lab_3/task1.asm Lab_3/task1.o
ld -o Lab_3/task1 Lab_3/task1.o
fasm Lab_3/task2.asm Lab_3/task2.o
ld -o Lab_3/task2 Lab_3/task2.o
gcc -Wall -Wextra -o Lab_3/task3 Lab_3/task3.c

./Lab_3/task1 A
./Lab_3/task2 1 2 7
./Lab_3/task3 1 2 7
```

Ожидаемый вывод: `65`, `1`, `1`. Для дополнительной сверки запустите обе программы вычисления с аргументами `1 2 1`: обе должны вывести `-1`.

## Вывод

Программы на FASM и C вычисляют одно и то же целочисленное выражение и при одинаковых входных данных дают одинаковый результат. Первая программа также корректно выводит ASCII-код символа, переданного параметром командной строки.